# League of Legends Match History 
**Author**: maxk678 | **Data Source:** [Riot API](https://developer.riotgames.com/)

This notebook demonstrates an end-toend data pipeline for a League of Legends player
1. **Data Collection**
2. **Data Enrichment**
3. **Storage**

> ⚠️ **Setup:** Requires a `.env` file with `riot_api_key`, `db_username`, `db_password`, `db_host`, and `db_port`.

---
## 1. Imports & API Setup

Load environment variables and define all API helper functions for the Riot Games API.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
import requests
import pandas as pd

api_key = os.environ.get('riot_api_key')

##Gets the puuid from riot_id and riot_tag
def get_puuid(gameName=None, tagLine=None, region='americas'):

    """
    Gets the puuid from riot_id and riot_tag

    Args:
        gameName (str,optional): Riot ID. Defualts to None.
        tagLine (str,optional): Riot Tag. Defualts to None.
        region (str,optional): Region. Defualts to 'americas'.
    Returns:
        str:puuid
    """
    root_url= f'https://{region}.api.riotgames.com/'
    endpoint= f'riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}'

    response = requests.get(root_url+endpoint+'?api_key='+api_key)

    return response.json()['puuid']

##Gets the riot_id and riot_tag from a puuid
def get_id_tag_from_puuid(puuid=None):
    """Gets the riot_id and riot_tag from a puuid

    Args:
        puuid (str, optional): puuid, Defualts to None.

    Returns:
        id (dict): Dictionary with riot_id and riot_tag
    """

    root_url = 'https://americas.api.riotgames.com/'
    endpoint = 'riot/account/v1/accounts/by-puuid/'

    response = requests.get(root_url + endpoint + puuid + '?api_key=' + api_key)

    id = {
        'gameName': response.json()['gameName'],
        'tagLine' : response.json()['tagLine']
    }

    return id

##Gets the top x players in soloq in NA1
def get_ladder(top=10):
    """Gets the top x players in soloq

    Args:
        top (int, optional): Number of players to return. Defaults to 300.
    Returns:
        DataFrame: Returns a DataFrame of the top x players in soloq
    """

    root_url = 'https://na1.api.riotgames.com/'
    challenger = 'lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5'
    grandmaster = 'lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5'
    master = 'lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'

    chall_resp = requests.get(root_url + challenger + '?api_key=' + api_key)
    chall_df = pd.DataFrame(chall_resp.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)

    gm_df = pd.DataFrame()
    m_df = pd.DataFrame()
    if top > 300:
            gm_resp = requests.get(root_url + grandmaster + '?api_key=' + api_key)
            gm_df = pd.DataFrame(gm_resp.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)
    if top > 1000:
            m_resp = requests.get(root_url + master + '?api_key=' + api_key)
            m_df = pd.DataFrame(m_resp.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)

    df = pd.concat([chall_df, gm_df, m_df]).reset_index(drop=True)[:top]

    df = df.reset_index().drop(columns=['rank']).rename(columns={'index':'rank'})
    df['rank'] += 1

    return df  

##Gets 50 most recent Match IDs from PUUID
def get_match_history(region=None, puuid=None, start=0, count=50):

    root_url = f'https://{region}.api.riotgames.com/'
    endpoint = f'lol/match/v5/matches/by-puuid/{puuid}/ids'
    query_params = f'?start={start}&count={count}'

    response = requests.get(root_url + endpoint + query_params + '&api_key=' + api_key)

    return response.json()

##Gets Match ID data from Match ID
def get_match_data_from_id(region=None, matchId=None):
    root_url = f'https://{region}.api.riotgames.com/'
    endpoint = f'/lol/match/v5/matches/{matchId}'    

    response = requests.get(root_url + endpoint + '?api_key=' + api_key)

    return response.json()

##Processes Match Data 
def process_match_json(match_json, puuid):
    ## Architecture
    metadata = match_json['metadata']
    info = match_json['info']
    players = info['participants']
    participants = metadata['participants']
    teams = info['teams']
    player = players[participants.index(puuid)]
    perks = player['perks']
    stats = perks['statPerks']
    styles = perks['styles']

    primary = styles[0]
    secondary = styles[1]

    side_dict = {
        100:'blue',
        200: 'red'
    }

    ## Information
    match_id = metadata['matchId']


    game_creation = info['gameCreation']
    game_duration = info['gameDuration']
    game_start_timestamp = info['gameStartTimestamp']
    game_end_timestamp = info['gameEndTimestamp']
    patch = info['gameVersion']
    game_mode = info['gameMode']

    riot_id = player['riotIdGameName']
    riot_tag = player['riotIdTagline']
    summoner_id = player['summonerId']
    summoner_name = player['summonerName']

    
    win = player['win']

    champ_id = player['championId']
    champ_level = player['champLevel']

    gold_earned = player['goldEarned']
    neutral_minions_killed = player['neutralMinionsKilled']
    total_minions_killed = player['totalMinionsKilled']

    kills = player['kills']
    deaths = player['deaths']
    assists = player['assists']
    first_blood = player['firstBloodKill']

    total_damage_dealt = player['totalDamageDealtToChampions']
    total_damage_shielded = player['totalDamageShieldedOnTeammates']
    total_damage_taken = player['totalDamageTaken']
    total_damage_healed = player['totalHealsOnTeammates']
    total_time_cc_dealt = player['totalTimeCCDealt']

    early_surrender = player['gameEndedInEarlySurrender']
    surrender = player['gameEndedInSurrender']

    item0 = player['item0']
    item1 = player['item1']
    item2 = player['item2']
    item3 = player['item3']
    item4 = player['item4']
    item5 = player['item5']
    item6 = player['item6']

    summoner_1_id = player['summoner1Id']
    summoner_2_id = player['summoner2Id']



    defense = stats['defense']
    flex = stats['flex']
    offense = stats['offense']

    primary_style = primary['style']
    secondary_style = primary['style']

    primary_keystone = primary['selections'][0]['perk']
    primary_perk_1 = primary['selections'][1]['perk']
    primary_perk_2 = primary['selections'][2]['perk']
    primary_perk_3 = primary['selections'][3]['perk']

    secondary_perk_1 = secondary['selections'][0]['perk']
    secondary_perk_2 = secondary['selections'][1]['perk']

    objectives_stolen = player['objectivesStolen']
    objectives_stolen_assists = player['objectivesStolenAssists']


    matchDF = pd.DataFrame({
        'match_id' : [match_id],
        'participants' :[participants],
        'game_creation' : [game_creation],
        'game_duration' : [game_duration],
        'game_start_timestamp' : [game_start_timestamp],
        'game_end_timestamp': [game_end_timestamp],
        'game_mode' : [game_mode],
        'patch' : [patch],
        'puuid' : [puuid],
        'riot_id' : [riot_id],
        'riot_tag' : [riot_tag],
        'summoner_id' : [summoner_id],
        'summoner_name' : [summoner_name],
        'win' : [win],
        'champion' : [champ_id],
        'champion_level' : [champ_level],
        'kills' : [kills],
        'deaths' : [deaths],
        'assists' : [assists],
        'summoner_1_id' : [summoner_1_id],
        'summoner_2_id' : [summoner_2_id],
        'gold_earned' : [gold_earned],
        'total_minions_killed' : [total_minions_killed],
        'total_neutral_minions_killed' : [neutral_minions_killed],
        'early_surrender' : [early_surrender],
        'surrender' : [surrender],
        'first_blood' : [first_blood],
        'objectives_stolen' : [objectives_stolen],
        'objectives_stolen_assists' : [objectives_stolen_assists],
        'total_damage_dealt_champions' : [total_damage_dealt],
        'total_damage_taken' : [total_damage_taken],
        'total_damage_shielded_teammates' : [total_damage_shielded],
        'total_heals_teammates' : [total_damage_healed],
        'total_time_crowd_controlled' : [total_time_cc_dealt],
        'item0' : [item0],
        'item1' : [item1],
        'item2' : [item2],
        'item3' : [item3],
        'item4' : [item4],
        'item5' : [item5],
        'item6' : [item6],
        'perk_keystone' : [primary_keystone],
        'perk_primary_row_1' : [primary_perk_1],
        'perk_primary_row_2' : [primary_perk_2],
        'perk_primary_row_3' : [primary_perk_3],
        'perk_secondary_row_1' : [secondary_perk_1],
        'perk_secondary_row_2' :[secondary_perk_2],
        'perk_primary_style' : [primary_style],
        'perk_secondary_style' : [secondary_style],
        'perk_shard_defense' : [defense],
        'perk_shard_flex' : [flex],
        'perk_shard_offense' : [offense],
    })

    return matchDF

---
# 2. Player Lookup

Verify the API connection by looking up the target player's PUUID (a unique persistent player identifier) and reversing the lookup to confirm the round-trip works correctly

In [2]:
ladder = get_ladder(top=300)
ladder[['rank', 'puuid', 'leaguePoints', 'wins', 'losses']]

,rank,puuid,leaguePoints,wins,losses
0,1,ih9_0UsD72F5_2ZsSTGo67NdkEPsT24ReE8ixxM7INxXaN...,2507,307,185
1,2,OKkwOLkb0X0kDK8HmroYpxmSB8Yvn9GojxHHhQZfJRsQUU...,2385,142,53
2,3,WQ4lcDQH2xb5KggyiZ3fbgjmSjD880yigUZrr0TD2s_y-D...,2294,256,153
3,4,UjhzdUaIYqKx_bQLzlKkT6i4slo3o2OspAxlp_-81FQAX4...,2270,185,113
4,5,sPSCdjYElrp7CmztZUzJPSSwpB0tzWV3rzrVpg3XxeHNQv...,2248,206,112
...,...,...,...,...,...
295,296,2Fqju1d5cSfbyWKrH7EylN_3julLIcEbAdfJ5BLF1WgpYK...,1004,179,186
296,297,dzBlmJXIqvLMnLwJzJ5ucmdIAsq8LXmQKveLZ54-ktmaA7...,1004,84,66
297,298,S3HxFv2nIvAdVfC_00vr5DdLI4tsnVKw-n9oYza1OUZJ9R...,1003,162,125
298,299,V5DM5KOHrUDrydy5uDFKWK4mMjdigp7EbSFGF-1uDc44EB...,992,138,128


In [8]:
# Option A — pick a player from the ladder above by index (0 = rank 1)
puuid = ladder['puuid'].iloc[150]

# Option B - look up a specific player by Riot ID
#GAME_NAME = 'TaricMidGG'
#TAG_LINE = 'NA1'
REGION = 'americas'
#puuid = get_puuid(gameName=GAME_NAME, tagLine=TAG_LINE, region=REGION)

#Prints the PUUID of the option you picked
print(f'PUUID: {puuid}')

PUUID: OqoZg-JwNCp9NZCaHjwoKVtNkCj3nDSh1ELzDRkija4PWp7o0i42-KuQo4cr2f9dk3kNz9uYzMcjwg


In [9]:
# Reverse lookup - confirm PUUID maps back to the correct Riot ID
get_id_tag_from_puuid(puuid=puuid)

{'gameName': 'Exuan', 'tagLine': 'xyi'}

---
# 3. Fetch Match History

Retrieve the 50 most recent match IDs for the player, then loop through each to fetch mutch match data and extract per-game stats into a single DataFrame.

In [12]:
# Fetch most recent match IDs
match_ids = get_match_history(region=REGION, puuid=puuid)
print(f"Fetched {len(match_ids)} match IDs")
match_ids[:50]

Fetched 50 match IDs


['NA1_5515713664',
 'NA1_5515146610',
 'NA1_5515117526',
 'NA1_5515107787',
 'NA1_5515091837',
 'NA1_5515071504',
 'NA1_5515060850',
 'NA1_5515048614',
 'NA1_5515027392',
 'NA1_5515007702',
 'NA1_5513694325',
 'NA1_5512166086',
 'NA1_5512132446',
 'NA1_5512113233',
 'NA1_5512077143',
 'NA1_5510421236',
 'NA1_5510288646',
 'NA1_5510283987',
 'NA1_5510274363',
 'NA1_5510266908',
 'NA1_5510249023',
 'NA1_5510217661',
 'NA1_5510208834',
 'NA1_5510183064',
 'NA1_5507893819',
 'NA1_5507767659',
 'NA1_5507332794',
 'NA1_5507288090',
 'NA1_5506892018',
 'NA1_5506707647',
 'NA1_5506678400',
 'NA1_5506236034',
 'NA1_5505169440',
 'NA1_5504622040',
 'NA1_5504598280',
 'NA1_5504564854',
 'NA1_5504224672',
 'NA1_5504219243',
 'NA1_5500379250',
 'NA1_5500339702',
 'NA1_5500319527',
 'NA1_5500295264',
 'NA1_5498832501',
 'NA1_5498810320',
 'NA1_5498788331',
 'NA1_5498767758',
 'NA1_5498738910',
 'NA1_5497396072',
 'NA1_5496597464',
 'NA1_5496589736']

In [11]:
# Loop through match IDs and build the DataFrame
df = pd.DataFrame()
for match_id in match_ids:
    game = get_match_data_from_id(region=REGION, matchId=match_id)
    matchDF = process_match_json(game, puuid=puuid)
    df = pd.concat([df, matchDF])

df = df.reset_index(drop=True)
print(f"Shape: {df.shape}")
df.head()

Shape: (50, 52)


,match_id,participants,game_creation,game_duration,game_start_timestamp,game_end_timestamp,game_mode,patch,puuid,riot_id,...,perk_primary_row_1,perk_primary_row_2,perk_primary_row_3,perk_secondary_row_1,perk_secondary_row_2,perk_primary_style,perk_secondary_style,perk_shard_defense,perk_shard_flex,perk_shard_offense
0,NA1_5515713664,[6Z727IM8ZU2zzKgliuy5kUiVsUk6ZJGnmP_2odsTyV6RO...,1773942812780,1125,1773942830022,1773943955514,CLASSIC,16.6.755.2788,OqoZg-JwNCp9NZCaHjwoKVtNkCj3nDSh1ELzDRkija4PWp...,Exuan,...,9111,9105,8014,8143,8135,8000,8000,5001,5008,5008
1,NA1_5515146610,[TtxWPU868ItXAATbyQyGmUR2TEEyL5SKpjKE2SIMGdep9...,1773870307827,1659,1773870331421,1773871990069,CLASSIC,16.6.753.8272,OqoZg-JwNCp9NZCaHjwoKVtNkCj3nDSh1ELzDRkija4PWp...,Exuan,...,9111,9105,8014,8143,8135,8000,8000,5001,5008,5008
2,NA1_5515117526,[zx8mwpODga63_OZ9U0mN2FIQyOTDBy4dX6BefxyRSQbuK...,1773866804352,2286,1773866832356,1773869118573,CLASSIC,16.6.753.8272,OqoZg-JwNCp9NZCaHjwoKVtNkCj3nDSh1ELzDRkija4PWp...,Exuan,...,9111,9105,8014,8143,8135,8000,8000,5001,5008,5008
3,NA1_5515107787,[IXJrV4YnvqmqRa6fZEiHwl26syIDZhGJjUiISlAppVDOm...,1773865065289,1163,1773865082732,1773866246263,CLASSIC,16.6.753.8272,OqoZg-JwNCp9NZCaHjwoKVtNkCj3nDSh1ELzDRkija4PWp...,Exuan,...,8143,8137,8135,9104,8014,8100,8100,5001,5008,5008
4,NA1_5515091837,[SuguWolGD7_s_ICcTqEXcXObIKc9w3ifxLMP91Qvv1U1G...,1773863165018,1592,1773863183487,1773864775679,CLASSIC,16.6.753.8272,OqoZg-JwNCp9NZCaHjwoKVtNkCj3nDSh1ELzDRkija4PWp...,Exuan,...,8446,8473,8451,8345,8304,8400,8400,5001,5008,5008


---
# Data Enrichment - ID to Name Mapping 

Raw match data stores, perks, items, and champions as numeric IDs. Use [Community Dragon](https://communitydragon.org/) - an open data source for League assets - to map these IDs to readable names. 

In [13]:
perk = 'https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/perks.json'
perk_styles = 'https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/perkstyles.json'
items = 'https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/items.json'
champ_summary = 'https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/champion-summary.json'

perk_json = requests.get(perk).json()
perk_styles_json = requests.get(perk_styles).json()
items_json = requests.get(items).json()
champ_summary_json = requests.get(champ_summary).json()

In [14]:
def json_extract(obj, key):

    arr = []

    def extract (obj, arr, key):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == key:
                    arr.append(v)
                elif isinstance(v,(dict, list)):
                    extract(v, arr, key)
        elif isinstance(obj, list):
            for item in obj:
                extract (item, arr, key)

        return arr
    
    values = extract(obj, arr, key)
    return values

In [16]:
##Perk dict
perk_ids = json_extract(perk_json,'id')
perk_names = json_extract(perk_json,'name')

perk_dict = {int(i): j for i, j in zip(perk_ids, perk_names)}

perk_columns = [
    'perk_keystone', 'perk_primary_row_1', 'perk_primary_row_2', 'perk_primary_row_3',
    'perk_secondary_row_1', 'perk_secondary_row_2', 'perk_primary_style', 'perk_secondary_style'
]

for col in perk_columns:
    df[col] = df[col].map(perk_dict).fillna(df[col])


##Perk Style Dict
perk_styles_ids = json_extract(perk_styles_json,'id')
perk_styles_names = json_extract(perk_styles_json,'name')

perk_styles_dict = {style['id']: style['name'] for style in perk_styles_json['styles']}

style_columns = ['perk_primary_style', 'perk_secondary_style']

for col in style_columns:
    df[col] = df[col].map(perk_styles_dict).fillna(df[col])

##Shard Dict
shard_dict = {
    5001: 'Health Scaling',
    5002: 'Armor',
    5003: 'Magic Resist',
    5005: 'Attack Speed',
    5007: 'Ability Haste',
    5008: 'Adaptive Force',
    5010: 'Move Speed',
    5011: 'Health',
    5013: 'Tenacity'
}

shard_columns = ['perk_shard_defense', 'perk_shard_flex', 'perk_shard_offense']

for col in shard_columns:
    df[col] = df[col].map(shard_dict).fillna(df[col])

##Items dict
items_dict = {item['id']: item['name'] for item in items_json}

item_columns = ['item0', 'item1', 'item2', 'item3', 'item4', 'item5', 'item6']

for col in item_columns:
    df[col] = df[col].map(items_dict).fillna(df[col])

##Champ Dict
champion_dict = {champ['id']: champ['name'] for champ in champ_summary_json}

df['champion'] = df['champion'].map(champion_dict).fillna(df['champion'])


Preview the enriched DataFrame

In [17]:
pd.set_option('display.max_columns', None)
df.head()

,match_id,participants,game_creation,game_duration,game_start_timestamp,game_end_timestamp,game_mode,patch,puuid,riot_id,riot_tag,summoner_id,summoner_name,win,champion,champion_level,kills,deaths,assists,summoner_1_id,summoner_2_id,gold_earned,total_minions_killed,total_neutral_minions_killed,early_surrender,surrender,first_blood,objectives_stolen,objectives_stolen_assists,total_damage_dealt_champions,total_damage_taken,total_damage_shielded_teammates,total_heals_teammates,total_time_crowd_controlled,item0,item1,item2,item3,item4,item5,item6,perk_keystone,perk_primary_row_1,perk_primary_row_2,perk_primary_row_3,perk_secondary_row_1,perk_secondary_row_2,perk_primary_style,perk_secondary_style,perk_shard_defense,perk_shard_flex,perk_shard_offense
0,NA1_5515713664,[6Z727IM8ZU2zzKgliuy5kUiVsUk6ZJGnmP_2odsTyV6RO...,1773942812780,1125,1773942830022,1773943955514,CLASSIC,16.6.755.2788,OqoZg-JwNCp9NZCaHjwoKVtNkCj3nDSh1ELzDRkija4PWp...,Exuan,xyi,jIXaz6Z8vI_ExlDk1doQ6us1qPZ8LI6MVxbTPatxgHLU8Dw,,False,Kayn,11,3,4,0,4,11,7382,16,115,False,True,True,0,0,4520,13738,0,0,217,Voltaic Cyclosword,Black Cleaver,0,0,0,Boots,Oracle Lens,Conqueror,Triumph,Legend: Haste,Coup de Grace,Sudden Impact,Treasure Hunter,Precision,Precision,Health Scaling,Adaptive Force,Adaptive Force
1,NA1_5515146610,[TtxWPU868ItXAATbyQyGmUR2TEEyL5SKpjKE2SIMGdep9...,1773870307827,1659,1773870331421,1773871990069,CLASSIC,16.6.753.8272,OqoZg-JwNCp9NZCaHjwoKVtNkCj3nDSh1ELzDRkija4PWp...,Exuan,xyi,jIXaz6Z8vI_ExlDk1doQ6us1qPZ8LI6MVxbTPatxgHLU8Dw,,True,Kayn,17,15,1,5,4,11,15558,40,187,False,True,True,0,0,25196,31578,0,0,346,Hubris,Hexdrinker,Tunneler,Black Cleaver,Death's Dance,Plated Steelcaps,Oracle Lens,Conqueror,Triumph,Legend: Haste,Coup de Grace,Sudden Impact,Treasure Hunter,Precision,Precision,Health Scaling,Adaptive Force,Adaptive Force
2,NA1_5515117526,[zx8mwpODga63_OZ9U0mN2FIQyOTDBy4dX6BefxyRSQbuK...,1773866804352,2286,1773866832356,1773869118573,CLASSIC,16.6.753.8272,OqoZg-JwNCp9NZCaHjwoKVtNkCj3nDSh1ELzDRkija4PWp...,Exuan,xyi,jIXaz6Z8vI_ExlDk1doQ6us1qPZ8LI6MVxbTPatxgHLU8Dw,,True,Kayn,18,13,8,12,4,11,18202,36,211,False,False,False,0,0,38979,46492,0,0,438,Black Cleaver,Hubris,Death's Dance,Maw of Malmortius,Serpent's Fang,Mercury's Treads,Oracle Lens,Conqueror,Triumph,Legend: Haste,Coup de Grace,Sudden Impact,Treasure Hunter,Precision,Precision,Health Scaling,Adaptive Force,Adaptive Force
3,NA1_5515107787,[IXJrV4YnvqmqRa6fZEiHwl26syIDZhGJjUiISlAppVDOm...,1773865065289,1163,1773865082732,1773866246263,CLASSIC,16.6.753.8272,OqoZg-JwNCp9NZCaHjwoKVtNkCj3nDSh1ELzDRkija4PWp...,Exuan,xyi,jIXaz6Z8vI_ExlDk1doQ6us1qPZ8LI6MVxbTPatxgHLU8Dw,,False,Kayn,11,5,6,1,4,11,8295,21,96,False,True,False,0,0,7903,15749,0,0,263,Serrated Dirk,Voltaic Cyclosword,Serpent's Fang,0,0,Ionian Boots of Lucidity,Farsight Alteration,Dark Harvest,Sudden Impact,Sixth Sense,Treasure Hunter,Legend: Alacrity,Coup de Grace,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
4,NA1_5515091837,[SuguWolGD7_s_ICcTqEXcXObIKc9w3ifxLMP91Qvv1U1G...,1773863165018,1592,1773863183487,1773864775679,CLASSIC,16.6.753.8272,OqoZg-JwNCp9NZCaHjwoKVtNkCj3nDSh1ELzDRkija4PWp...,Exuan,xyi,jIXaz6Z8vI_ExlDk1doQ6us1qPZ8LI6MVxbTPatxgHLU8Dw,,False,Jayce,15,0,9,1,4,12,9103,222,0,False,False,False,0,0,11373,25174,0,0,124,Doran's Blade,Eclipse,Muramana,Executioner's Calling,Long Sword,Plated Steelcaps,Stealth Ward,Grasp of the Undying,Demolish,Bone Plating,Overgrowth,Biscuit Delivery,Magical Footwear,Resolve,Resolve,Health Scaling,Adaptive Force,Adaptive Force


---
# 5. Export to PostgreSQL

Store the enriched match data to a PostgreSQL database using `pangres` — existing records are updated, new ones are added, so no duplicates build up on repeated runs.

Connection credentials are loaded from `.env` via `python-dotenv`.

In [34]:
##Creating connection to postgresql database
import os
from dotenv import load_dotenv
load_dotenv()

from pangres import upsert
from sqlalchemy import text, create_engine

db_username=os.environ.get('db_username')
db_password=os.environ.get('db_password')
db_host=os.environ.get('db_host')
db_port=os.environ.get('db_port')
db_name=os.environ.get('db_name')

def create_db_connection_string(db_username, db_password, db_host, db_port, db_name):
    connection_url = 'postgresql+psycopg2://'+db_username+':'+db_password+'@'+db_host+':'+db_port+'/'+db_name
    return connection_url

conn_url = create_db_connection_string(db_username, db_password, db_host, db_port, db_name)

db_engine = create_engine(conn_url, pool_recycle=3600)

connection = db_engine.connect()

In [35]:
## Create unique primary key from match_id and puuid
df['uuid'] = df['match_id'] + '_' + df['puuid']
df = df.set_index('uuid')

Confirm primary key creation

In [39]:
df.head()

,match_id,participants,game_creation,game_duration,game_start_timestamp,game_end_timestamp,game_mode,patch,puuid,riot_id,riot_tag,summoner_id,summoner_name,win,champion,champion_level,kills,deaths,assists,summoner_1_id,summoner_2_id,gold_earned,total_minions_killed,total_neutral_minions_killed,early_surrender,surrender,first_blood,objectives_stolen,objectives_stolen_assists,total_damage_dealt_champions,total_damage_taken,total_damage_shielded_teammates,total_heals_teammates,total_time_crowd_controlled,item0,item1,item2,item3,item4,item5,item6,perk_keystone,perk_primary_row_1,perk_primary_row_2,perk_primary_row_3,perk_secondary_row_1,perk_secondary_row_2,perk_primary_style,perk_secondary_style,perk_shard_defense,perk_shard_flex,perk_shard_offense
uuid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
NA1_5490303087_t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNmY9UTTO_x1b4Ddt_lhOfHrI4nrSjQFwpw,NA1_5490303087,[oXOnGvsfHrMrBDxweAq6tU7MB5kSwOFp157XCSdDvX702...,1770932713050,1242,1770932742218,1770933984645,ARAM,16.3.745.7600,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Taliyah,18,9,8,24,4,6,13917,50,0,False,False,False,0,0,28017,18248,0,0,360,Luden's Echo,Rabadon's Deathcap,Sorcerer's Shoes,Blackfire Torch,Stormsurge,0,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Mementos,Treasure Hunter,Coup de Grace,Presence of Mind,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
NA1_5487109265_t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNmY9UTTO_x1b4Ddt_lhOfHrI4nrSjQFwpw,NA1_5487109265,[t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBN...,1770582421604,1723,1770582437192,1770584160231,ARAM,16.3.744.7656,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,False,Twisted Fate,18,12,11,26,4,6,18975,93,0,False,False,True,0,0,65596,35896,0,0,73,Blackfire Torch,Riftmaker,Ionian Boots of Lucidity,Liandry's Torment,Shadowflame,Rabadon's Deathcap,Poro-Snax,Arcane Comet,Manaflow Band,Transcendence,Scorch,Coup de Grace,Presence of Mind,Sorcery,Sorcery,Health Scaling,Adaptive Force,Adaptive Force
NA1_5478122035_t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNmY9UTTO_x1b4Ddt_lhOfHrI4nrSjQFwpw,NA1_5478122035,[qJstPPER4-qL0bQLlqUlh7Us_7LSaysoV90xzwhZ9A9Az...,1769697961312,493,1769698090757,1769698583532,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,False,Twitch,10,8,5,4,4,6,8274,18,0,False,True,False,0,0,10994,7230,0,0,44,Yun Tal Wildarrows,The Collector,Berserker's Greaves,0,0,0,0,Lethal Tempo,Triumph,Legend: Alacrity,Coup de Grace,Taste of Blood,Treasure Hunter,Precision,Precision,Health Scaling,Adaptive Force,Attack Speed
NA1_5477493562_t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNmY9UTTO_x1b4Ddt_lhOfHrI4nrSjQFwpw,NA1_5477493562,[6KWEvXJA31CCUQ77fR-E7QwqyHG_72uEaUTKjnxWHHjFK...,1769638816377,995,1769638836432,1769639830844,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,False,Malzahar,15,5,10,25,4,3,10787,54,0,False,False,False,0,0,27740,25454,0,0,269,Blackfire Torch,Rylai's Crystal Scepter,Liandry's Torment,Sorcerer's Shoes,Amplifying Tome,0,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Mementos,Ultimate Hunter,Coup de Grace,Presence of Mind,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
NA1_5477456755_t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNmY9UTTO_x1b4Ddt_lhOfHrI4nrSjQFwpw,NA1_5477456755,[BK4-FEC7sudVe5ysZaGMi8Kkil7rprZ4FacCT_oarNtBz...,1769635418714,1003,1769635456616,1769636459818,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Vel'Koz,16,9,7,15,4,7,12009,34,0,False,False,False,0,0,31226,17458,0,508,197,Malignance,Refillable Potion,Liandry's Torment,Sorcerer's Shoes,Oblivion Orb,Shadowflame,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Me

In [40]:
# Upsert into PostgreSQL
upsert(con=connection, df=df, schema='soloq', table_name='regional_player_matches', create_table=True, create_schema=True, if_row_exists='update')

In [41]:
connection.commit()

---
# 6. Verify Export - Sample Query
Read back from the database to confirm the data landed correctly. As a quick sanity check, filter for games where the player won.

In [42]:
with db_engine.connect() as connection: 
    df = pd.read_sql(text(f"""select 
                                * 
                            from 
                                soloq.regional_player_matches
                            where
                                win = 'True'"""), connection)

In [43]:
df.head()

,uuid,match_id,participants,game_creation,game_duration,game_start_timestamp,game_end_timestamp,game_mode,patch,puuid,riot_id,riot_tag,summoner_id,summoner_name,win,champion,champion_level,kills,deaths,assists,summoner_1_id,summoner_2_id,gold_earned,total_minions_killed,total_neutral_minions_killed,early_surrender,surrender,first_blood,objectives_stolen,objectives_stolen_assists,total_damage_dealt_champions,total_damage_taken,total_damage_shielded_teammates,total_heals_teammates,total_time_crowd_controlled,item0,item1,item2,item3,item4,item5,item6,perk_keystone,perk_primary_row_1,perk_primary_row_2,perk_primary_row_3,perk_secondary_row_1,perk_secondary_row_2,perk_primary_style,perk_secondary_style,perk_shard_defense,perk_shard_flex,perk_shard_offense
0,NA1_5490303087_t6fDViYubCso_U1bup1dVyvndQ-FBtQ...,NA1_5490303087,[oXOnGvsfHrMrBDxweAq6tU7MB5kSwOFp157XCSdDvX702...,1770932713050,1242,1770932742218,1770933984645,ARAM,16.3.745.7600,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Taliyah,18,9,8,24,4,6,13917,50,0,False,False,False,0,0,28017,18248,0,0,360,Luden's Echo,Rabadon's Deathcap,Sorcerer's Shoes,Blackfire Torch,Stormsurge,0,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Mementos,Treasure Hunter,Coup de Grace,Presence of Mind,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
1,NA1_5477456755_t6fDViYubCso_U1bup1dVyvndQ-FBtQ...,NA1_5477456755,[BK4-FEC7sudVe5ysZaGMi8Kkil7rprZ4FacCT_oarNtBz...,1769635418714,1003,1769635456616,1769636459818,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Vel'Koz,16,9,7,15,4,7,12009,34,0,False,False,False,0,0,31226,17458,0,508,197,Malignance,Refillable Potion,Liandry's Torment,Sorcerer's Shoes,Oblivion Orb,Shadowflame,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Mementos,Ultimate Hunter,Manaflow Band,Transcendence,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
2,NA1_5477429141_t6fDViYubCso_U1bup1dVyvndQ-FBtQ...,NA1_5477429141,[bG4SL-lDiupFvX3OdSudW9ASN17DIlORufSZ8PdU4hHCz...,1769632965993,1777,1769633030304,1769634807449,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Pyke,18,21,16,26,4,32,18995,20,0,False,True,False,0,0,36075,57344,0,0,184,Youmuu's Ghostblade,Mercury's Treads,Axiom Arc,The Collector,Edge of Night,Hubris,Poro-Snax,Dark Harvest,Sudden Impact,Grisly Mementos,Ultimate Hunter,Coup de Grace,Presence of Mind,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
3,NA1_5475838808_t6fDViYubCso_U1bup1dVyvndQ-FBtQ...,NA1_5475838808,[0pmbF_Tv0EPBvPEqoFVWGXw3GI3Do_2iw0aI4v6diyyVZ...,1769473407319,1363,1769473491530,1769474854529,ARAM,16.2.740.1491,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Shaco,18,7,11,35,4,32,14388,25,0,False,False,False,0,0,31738,30879,0,0,266,Luden's Echo,Liandry's Torment,Sorcerer's Shoes,Blackfire Torch,Shadowflame,Oblivion Orb,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Mementos,Ultimate Hunter,Manaflow Band,Transcendence,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
4,NA1_5475820811_t6fDViYubCso_U1bup1dVyvndQ-FBtQ...,NA1_5475820811,[sDC9hdruHEhgpWh-buRQIXN-TeQcW550xDv13Xq7T4raC...,1769472401176,744,1769472430144,1769473174560,ARAM,16.2.740.1491,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Corki,14,13,4,19,4,32,10451,27,0,False,True,True,0,0,19148,11361,0,0,0,Trinity Force,Muramana,Long Sword,Ionian Boots of Lucidity,Pickaxe,Long Sword,Poro-Snax,Dark Harvest,Taste of Blood,Grisly Mementos,Treasure Hunter,Coup de Grace,Presence of Mind,Domination,Domination,Health Scaling,Adaptive Force,Attack Speed
